# Number to Words Converter

Convert a number extracted from a line of text into its English word representation. Supports up to billions.

## Lookup Tables

English number names are irregular (eleven, twelve, thirteen... not "oneteen"), so lookup tables are the way to go.

In [ ]:
# empty strings so index matches the number, teens are irregular so we store them directly
ONES = [
    "", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine",
    "ten", "eleven", "twelve", "thirteen", "fourteen", "fifteen", "sixteen",
    "seventeen", "eighteen", "nineteen",
]

TENS = [
    "", "", "twenty", "thirty", "forty", "fifty", "sixty", "seventy", "eighty", "ninety",
]

## Converter Functions

In [ ]:
def convert_below_hundred(n):
    """Convert 0-99 to words."""
    if n < 20:
        return ONES[n]
    tens_part = TENS[n // 10]
    ones_part = ONES[n % 10]
    return f"{tens_part}-{ones_part}" if ones_part else tens_part


def convert_below_thousand(n):
    """Convert 0-999 to words."""
    if n < 100:
        return convert_below_hundred(n)
    hundreds_digit = n // 100
    remainder = n % 100
    result = f"{ONES[hundreds_digit]} hundred"
    if remainder:
        result += f" and {convert_below_hundred(remainder)}"
    return result


def convert(n):
    """Convert a non-negative integer to English words."""
    if n == 0:
        return "zero"

    groups = []
    while n > 0:
        groups.append(n % 1000)
        n //= 1000

    SCALES = ["", "thousand", "million", "billion", "trillion"]

    parts = []
    for group, scale in zip(groups, SCALES):
        if group != 0:
            word = convert_below_thousand(group)
            parts.append(f"{word} {scale}".strip())

    parts.reverse()

    if len(parts) > 1 and 0 < groups[0] < 100:
        return ', '.join(parts[:-1]) + ' and ' + parts[-1]

    return ', '.join(parts)

In [ ]:
{n: convert(n) for n in [0, 1, 11, 20, 42, 99, 100, 536, 1000, 1001, 9121, 10022]}

## Number Extraction

Extract exactly one standalone number from a line of text. Reject lines with multiple digit groups or numbers glued to special characters.

In [ ]:
import re

# number must be standalone — not glued to special characters like #
STANDALONE_NUMBER = re.compile(r"(?:^|(?<=\s))(\d+)(?=\s|[.,;:!?]|$)")


def extract_number(line):
    """Extract a single valid integer from a line, or None if invalid."""
    all_digits = re.findall(r"\d+", line)
    if len(all_digits) != 1:
        return None

    match = STANDALONE_NUMBER.search(line)
    if not match:
        return None

    return int(match.group(1))

## Process Line

Wire extraction and conversion together.

In [ ]:
def process_line(line):
    number = extract_number(line)
    if number is None:
        return "number invalid"
    return convert(number)

## Verification

In [ ]:
test_lines = [
    "The pump is 536 deep underground.",
    "We processed 9121 records.",
    "Variables reported as having a missing type #65678.",
    "Interactive and printable 10022 ZIP code.",
    "The database has 66723107008 records.",
    "I received 23 456,9 KGs.",
]

expected_outputs = [
    "five hundred and thirty-six",
    "nine thousand, one hundred and twenty-one",
    "number invalid",
    "ten thousand and twenty-two",
    "sixty-six billion, seven hundred and twenty-three million, one hundred and seven thousand and eight",
    "number invalid",
]

results = {line: process_line(line) for line in test_lines}

for (line, result), expected in zip(results.items(), expected_outputs):
    assert result == expected, f"FAIL: got '{result}', expected '{expected}'"

results

## File Input

In [ ]:
with open("input.txt") as f:
    lines = [l.strip() for l in f if l.strip()]

[process_line(line) for line in lines]